In [ ]:
import numpy as np
from openai import OpenAI
import os
import json
from pprint import pprint

In [ ]:
#  Documents (knowledge base) to be used for retrieval
docs = [
    "LangChain is a framework for developing LLM applications.",
    "RAG fetches relevant external context to minimize hallucinations.",
    "Fine-tuning adapts model weights on domain-specific datasets.",
]

In [ ]:
client = OpenAI(api_key="paste your key here")


In [ ]:
def get_embeddings(texts):
    response = client.embeddings.create(
        input=texts,
        model="text-embedding-3-small"
    )
    return np.array(response.data[0].embedding)


In [ ]:
docs_vectors = [get_embeddings([doc]) for doc in docs]


In [ ]:

# 3. Simple Cosine Similarity Search manually implemented
def search(query, top_k=1):
    q_vec = get_embeddings(query)
    scores = [
        np.dot(q_vec, d_vec)
        / (np.linalg.norm(q_vec) * np.linalg.norm(d_vec))
        for d_vec in docs_vectors
    ]
    best_indices =np.argmax (scores)
    print(np.argmax(scores))
    print(scores)   
    return docs[best_indices]
    

In [ ]:
query = "What is electron?"
result = search(query)
result


In [ ]:
prompt=f"context: {result}\n\nquestion: {query}\n\n answer the question based on the context provided. If the answer is not contained within the context, say 'I don't know.'"

In [ ]:
# llm call to get the answer based on the context retrieved
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

In [ ]:

print(response.choices[0].message.content)